# Measurement Error

Our goal is to see how measurement error affects the observed estimates of a model.

## Setting up the notebook

Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import bernoulli
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

Configuration

In [ ]:
N_SAMPLES = 5000
N_FEATURES = 3
ERROR_P = 0.1  # Probability of being affected by measurement error
ERROR_PCT = 0.5  # 50% Increase in target (y)
RANDOM_STATE = 42

## Data Generating Process

Normally, we don't know the underlying data generating process (DGP) that created the
data (if we did, there would be no point in estimating it at all!).

However, in this notebook, we will generate a synthetic dataset. This means we will know
the form of the underlying DGP. We do this to be able to see how measurement error
affects our estimates by comparing the population parameters against the observed ones.

In [ ]:
# Generate IDs
ids = np.arange(N_SAMPLES) + 1

# Init a bernoulli RV and sample from it
brv = bernoulli(ERROR_P)
exclusions = brv.rvs(size=N_SAMPLES, random_state=RANDOM_STATE)

# Create a dataset of IDs and exclusion status
df_ids = pd.DataFrame({'id': ids, 'exclude': exclusions})

# View dataframe
print('Share of excluded observations:', df_ids['exclude'].mean())
df_ids.head(5)

Share of excluded observations: 0.0958


,id,exclude
0,1,0
1,2,1
2,3,0
3,4,0
4,5,0


Now we will generate a dataset with a known DGP. We will join both datasets to learn
how joins are performed, but also to synthetically create measurement error for all
rows marked with `exclude == 1`.


In [ ]:
# Generate data using sklearn
X, y, true_coef = make_regression(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_FEATURES,
    n_targets=1,
    bias=50.0,
    noise=1.0,
    shuffle=False,
    coef=True,  # Return population coefficients
    random_state=RANDOM_STATE
)
print('Population parameters:', true_coef)

# Turn X and y into a single dataframe
df_obs = pd.concat(
    objs=[
        pd.DataFrame(ids, columns=['id']),
        pd.DataFrame(X, columns=['x1', 'x2', 'x3']),
        pd.DataFrame(y, columns=['y'])
    ],
    axis=1
)

# View second dataset
df_obs.head()

Population parameters: [38.95952484  1.51074456 89.82730651]


,id,x1,x2,x3,y
0,1,0.496714,-0.138264,0.647689,128.509836
1,2,1.523030,-0.234153,-0.234137,88.042640
2,3,1.579213,0.767435,-0.469474,69.836189
3,4,0.542560,-0.463418,-0.465730,27.164924
4,5,0.241962,-1.913280,-1.724918,-98.871512


Join both tables together on the key `'id'`.

In [ ]:
# Join
df = df_ids.merge(right=df_obs, how='inner', on=['id'])

# View data
df.head()

,id,exclude,x1,x2,x3,y
0,1,0,0.496714,-0.138264,0.647689,128.509836
1,2,1,1.523030,-0.234153,-0.234137,88.042640
2,3,0,1.579213,0.767435,-0.469474,69.836189
3,4,0,0.542560,-0.463418,-0.465730,27.164924
4,5,0,0.241962,-1.913280,-1.724918,-98.871512


We will now create two types of measurement error:
1. Error on a covariate
2. Error on the target

In [ ]:
# Mask that only preserves
mask = df['exclude'].eq(1)

# Distort x3 for entries that should be excluded
df['x3_bad'] = df['x3'].copy()
df.loc[mask, 'x3_bad'] = df.loc[mask, 'x3_bad'] * (1 + ERROR_PCT)

# Distort y for entries that should be excluded
df['y_bad'] = df['y'].copy()
df.loc[mask, 'y_bad'] = df.loc[mask, 'y_bad'] * (1 + ERROR_PCT)

# View some affected cases
df.loc[mask, ['id', 'x3', 'x3_bad', 'y', 'y_bad']].head()

,id,x3,x3_bad,y,y_bad
1,2,-0.234137,-0.351205,88.042640,132.063960
11,12,-1.220844,-1.831265,-99.226267,-148.839401
33,34,-0.420645,-0.630968,1.055273,1.582909
34,35,-0.161286,-0.241929,19.944067,29.916101
43,44,0.068563,0.102844,33.755995,50.633993


## Fitting models

Real parameters:
* 38.95952484
* 1.51074456
* 89.82730651

1. Clean dataset

In [ ]:
model_clean = LinearRegression(fit_intercept=True)
model_clean.fit(
    X=df[['x1', 'x2', 'x3']],
    y=df['y']
)
model_clean.coef_.round(3)

array([38.968,  1.475, 89.815])

2. All observations and bad target

In [ ]:
model_bady_all = LinearRegression(fit_intercept=True)
model_bady_all.fit(
    X=df[['x1', 'x2', 'x3']],
    y=df['y_bad']
)
model_bady_all.coef_.round(3)

array([40.701,  1.564, 94.553])

3. Filtered dataset and bad target

In [ ]:
model_bady_masked = LinearRegression(fit_intercept=True)
model_bady_masked.fit(
    X=df.loc[~mask, ['x1', 'x2', 'x3']],
    y=df.loc[~mask, 'y_bad']
)
model_bady_masked.coef_.round(3)

array([38.968,  1.481, 89.812])

4. All observations and bad features

In [ ]:
model_badx_all = LinearRegression(fit_intercept=True)
model_badx_all.fit(
    X=df[['x1', 'x2', 'x3_bad']],
    y=df['y']
)
model_badx_all.coef_.round(3)

array([38.906,  1.388, 83.792])

5. Filtered observations and bad features

In [ ]:
model_badx_masked = LinearRegression(fit_intercept=True)
model_badx_masked.fit(
    X=df.loc[~mask, ['x1', 'x2', 'x3_bad']],
    y=df.loc[~mask, 'y']
)
model_badx_masked.coef_.round(3)

array([38.968,  1.481, 89.812])

In conclusion, fitting a model on a dataset that has any degree of measurment errors
will distort the estimated parameters. This is known as measurement bias, and it is
really hard to spot in real life!

Nevertheless, you should always think critically and use your best judgement to filter
out observations that you think have a non-negligible degree of measurment error.